In [ ]:
!pip install -U langchain langgraph fastapi uvicorn
#!pip install -U langchain-core langchain-community
#!pip install -U langchain-google-genai langchain-openai
!pip install -U "langchain[google-genai]"

In [ ]:
import os

if "SSLKEYLOGFILE" in os.environ:
  del os.environ["SSLKEYLOGFILE"]

import ssl
import _ssl  # noqa: F401

_ = ssl.OPENSSL_VERSION  # Force initialization


from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver,InMemorySaver
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os
import getpass
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")
#llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")
# 1. Define Graph & Memory (Same as above)
#llm = ChatOpenAI(model="gpt-4o")
#checkpointer = MemorySaver()  # Use PostgresSaver for production
checkpointer = InMemorySaver()

def call_model(state: MessagesState):
    return {"messages": llm.invoke(state["messages"])}


builder = StateGraph(MessagesState)
builder.add_node("chatbot", call_model)
builder.add_edge(START, "chatbot")
#graph = builder.compile()
graph = builder.compile(checkpointer=checkpointer)

# 2. Define Request Schema
class ChatRequest(BaseModel):
    message: str
    thread_id: str = "default_thread"


# 3. Create Endpoint
app = FastAPI()


@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    try:
        # Prepare input state
        input_state = {"messages": [HumanMessage(content=request.message)]}

        # Prepare config for state persistence
        config = {"configurable": {"thread_id": request.thread_id}}

        # Invoke Graph
        # Use ainvoke for non-blocking async execution
        result = await graph.ainvoke(input_state, config=config)

        # Extract the last message content
        last_message = result["messages"][-1].content
        return {"response": last_message, "thread_id": request.thread_id}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


if __name__ == "__main__":
    import uvicorn


    uvicorn.run("LanggraphAgentChat:app", host="127.0.0.1", port=8000,reload=True)


curl -X 'POST' \
  'https://langgraphagentchat.onrender.com/chat' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
  "message": "hi capital of INDIA",
  "thread_id": "default_thread"
}'

{
  "response": "The capital of India is **New Delhi**.",
  "thread_id": "default_thread"
}

In [ ]:
 curl.exe 'http://127.0.0.1:8000/chat' -X POST -H "Content-Type: application/json" -d '{\"message\": \"hi capital of INDIA\", \"thread_id\": \"default_thread\"}'

In [ ]:
deployment
push python files to github repository
create requirements.txt file with dependecies

build command
pip install -r Requirements.txt

start command
uvicorn LanggraphAgentChat:app --host 0.0.0.0 --port 8000

#uvicorn.run("LanggraphAgentChat:app", host="127.0.0.1", port=8000)

In [ ]:
 curl.exe https://langgraphchatbot.onrender.com/chat -X POST -H "Content-Type: application/json" -d '{\"message\": \"hi capital of INDIA\", \"thread_id\": \"default_thread\"}'

{"response":[{"type":"text","text":"Hello! The capital of India is **New Delhi**.","extras":{"signature":"EpMDCpADAWkUfRMXV5iTa4HwN/KFP2XR6aDeTfkhyP5nnE+M/7Ul94g7w3Ln0SAFWD8dF35RMW/ZS0urqaXMuJ8gvC5zpnMEI2N5r+dXejA55uMmLi5++j1B/6xKqWBQmUaSc3Jh23p8nwKcarA4iYLWJz5Abb821Ap8pD7Jv7Vy0eCanAnumQiIn1BCRQNT91dg5DcQV7j7Odqa/JPkoZQIePRjbTiFpKOlPKuEPNglYcKrhDzbDR5h/1IXsK9bmmlEx6x/67Dxgr/pyotSOLDDaH6R4gJ9sXNbB+7BXdJOaiFT3byF3IJSxeoghvmdwccSxzApzIuMmei/6LS5WjrsZV5rIvuW4vBV19HdHG8Y29OS3CNF8czGaVqPII4nEqTI29VzFd4LwJ3DdlyEDYc4q5ToTAzkUATl88dZf1M+08U5S2WkXjmWIKruSb6njVzjiOETg8MQJDhs8v77LxMnsF/sGmX6ASRHKZeq0gPfb1sdat0WGdTvrPhScAe1p+D79hmRc0NqAwxKdxywz22ELvhTmw=="}}],"thread_id":"default_thread"}